# 实验三：pandas 数据处理基础

<table style="margin: 0 auto; width: 30%; border-collapse: collapse; border: 1px solid black;" data-id="student-info">  
    <colgroup>  
        <col style="width: 35%;">  
        <col style="width: 65%;">  
    </colgroup>  
    <tr>  
        <td style="border: 1px solid black;">班级</td> <td style="border: 1px solid black;">25智能01</td>  
    </tr>  
    <tr>  
        <td style="border: 1px solid black;">学号</td> <td style="border: 1px solid black;">未填写</td>  
    </tr>  
    <tr>  
        <td style="border: 1px solid black;">姓名</td> <td style="border: 1px solid black;">未填写</td>  
    </tr>  
    <tr>  
        <td style="border: 1px solid black;">Email</td> <td style="border: 1px solid black;">123456@qq.com</td>  
    </tr>  
</table>

## 实验目标

本实验以一个小型零售业务的数据为主线，学习 pandas 中最常用、最重要的数据处理能力。完成实验后，你应该能够：

- 理解 `Series`、`DataFrame` 和索引的作用；
- 使用 `loc`、`iloc` 和布尔条件选择数据；
- 识别并处理缺失值、重复值和不统一的字符串；
- 使用分类类型和 `cut` 对连续数据分箱；
- 使用 `merge` 和 `concat` 组合多个数据表；
- 使用 `groupby` 完成分组汇总；
- 使用 `melt`、`pivot`、`stack` 和 `unstack` 改变数据形状。

实验内容选自《Python for Data Analysis（第 3 版）》第 5、7、8 章，并围绕实际数据处理流程重新组织。

## Jupyter Notebook 使用说明

1. 在右上角选择项目 `venv` 对应的 Python 内核。
2. 修改后先保存 Notebook，再运行测试。
3. 如果运行顺序混乱，请选择 **Restart Kernel and Run All**。
4. `...` 表示需要补充的代码；除习题函数外，不要修改测试和提交代码。

## 实验注意事项

1. 请在指定的地方按照实验指导要求来编写代码。
2. 请按照实验指导要求使用指定的变量名或函数名，不要使用其他的名字。
3. 不要添加任何额外的语句。
4. 不要添加任何额外的代码单元格。
5. 不要在不需要的地方修改作业代码，比如创建额外的变量，修改测试文件中的代码。
6. 实验指导中的`...`表示需要你补充代码的部分，其他部分的代码不用修改。
7. 代码提示中会给出估计的代码行数，例如大约1行代码，估计的代码行数只是一个参考值，实际编写时可能会有出入，请根据实际情况来编写。
8. 请独立完成作业，禁止抄袭，发现抄袭行为成绩记零分

## 1. 实验准备

In [1]:
# 导入依赖并初始化自动测试
%matplotlib inline

from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython import get_ipython

ip = get_ipython()
notebook_file = None
if ip is not None:
    notebook_file = ip.user_ns.get('__vsc_ipynb_file__')

if notebook_file is None:
    candidates = list(Path.cwd().glob('**/实验3-pandas.ipynb'))
    notebook_file = str(candidates[0]) if candidates else None

notebook_dir = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
project_root = notebook_dir.parent
data_dir = notebook_dir / 'data'

sys.path.append(str(project_root / 'tests'))
sys.path.append(str(project_root / 'util'))

from test_suite3 import TestSuite3
from notebook_info_extractor import extract_from_ipynb, is_valid_email

if notebook_file is None:
    raise FileNotFoundError('无法定位实验3-pandas.ipynb，请从项目目录中打开并运行本 Notebook。')

testsuite3 = TestSuite3()
stu_info = extract_from_ipynb(notebook_file)

if is_valid_email(stu_info.get('email')):
    print(f"stu_info: {stu_info}")
else:
    print(
        'Email地址无效，请在Notebook顶部的学生信息表中重新输入有效的Email地址，'
        '保存Notebook后重新运行本单元格。'
    )

stu_info: {'class_id': '25智能01', 'student_id': '未填写', 'name': '未填写', 'email': '123456@qq.com'}


### 1.1 实验数据

所有讲解和习题数据均来自 `data` 目录中的 CSV 文件：

| 文件 | 含义 | 主要字段 |
|---|---|---|
| `products.csv` | 商品主数据 | 商品编号、类别、价格、状态 |
| `orders.csv` | 原始订单 | 日期、客户、商品、数量、折扣 |
| `customers_dirty.csv` | 待清洗客户数据 | 姓名、城市、客户类型、邮箱 |
| `inventory_shanghai.csv` | 上海仓库存 | 商品、仓库、库存量 |
| `inventory_beijing.csv` | 北京仓库存 | 商品、仓库、库存量 |
| `monthly_sales_wide.csv` | 月度销售宽表 | 商品及各月份销售额 |

这些数据是为实验构造的小型教学数据，不代表真实业务。

In [2]:
products = pd.read_csv(data_dir / 'products.csv')
orders = pd.read_csv(data_dir / 'orders.csv')
customers_raw = pd.read_csv(data_dir / 'customers_dirty.csv')
inventory_shanghai = pd.read_csv(data_dir / 'inventory_shanghai.csv')
inventory_beijing = pd.read_csv(data_dir / 'inventory_beijing.csv')
monthly_sales_wide = pd.read_csv(data_dir / 'monthly_sales_wide.csv')

pd.DataFrame({
    '数据表': ['products', 'orders', 'customers_raw', 'inventory_shanghai',
             'inventory_beijing', 'monthly_sales_wide'],
    '行数': [len(products), len(orders), len(customers_raw), len(inventory_shanghai),
           len(inventory_beijing), len(monthly_sales_wide)],
    '列数': [products.shape[1], orders.shape[1], customers_raw.shape[1],
           inventory_shanghai.shape[1], inventory_beijing.shape[1], monthly_sales_wide.shape[1]],
})

,数据表,行数,列数
0,products,8,5
1,orders,13,6
2,customers_raw,8,5
3,inventory_shanghai,5,3
4,inventory_beijing,5,3
5,monthly_sales_wide,7,4


## 2. DataFrame、索引与数据选择

`Series` 是带索引的一维数据，`DataFrame` 是由多个共享行索引的 Series 组成的二维表格。索引不仅用于显示，还参与选择、对齐和运算。

最常用的选择方式是：

- `df['列名']`：选择单列，结果通常是 Series；
- `df[['列1', '列2']]`：选择多列，结果是 DataFrame；
- `df.loc[行标签, 列标签]`：按照标签选择；
- `df.iloc[行位置, 列位置]`：按照整数位置选择；
- 布尔条件：筛选满足条件的行。

比较既可以写成 `df['status'] == 'active'`，也可以使用等价的方法 `df['status'].eq('active')`。常用比较方法包括 `eq`（等于）、`ne`（不等于）、`lt`（小于）、`le`（小于等于）、`gt`（大于）和 `ge`（大于等于）。

组合多个条件时使用：

- `&` 表示“并且”；
- `|` 表示“或者”；
- `~` 表示“取反”。

每个比较条件都要放在括号中，不能用 Python 的 `and`、`or` 直接连接 Series。索引可能是整数时，优先显式使用 `loc` 或 `iloc`，避免把“标签”和“位置”混淆。

In [3]:
products.head(4)

,product_id,product_name,category,unit_price,status
0,P001,Wireless Mouse,Electronics,89,active
1,P002,Mechanical Keyboard,Electronics,329,active
2,P003,USB-C Hub,Electronics,199,active
3,P004,Notebook,Stationery,18,active


In [4]:
# loc 按标签或条件选择；多个条件必须分别加括号
products.loc[
    (products['status'].eq('active')) & (products['unit_price'].le(200)),
    ['product_id', 'product_name', 'category', 'unit_price'],
]

,product_id,product_name,category,unit_price
0,P001,Wireless Mouse,Electronics,89
2,P003,USB-C Hub,Electronics,199
3,P004,Notebook,Stationery,18
4,P005,Gel Pen Set,Stationery,25
6,P007,Water Bottle,Home,79


In [5]:
# iloc 按位置选择：前三行、前三列
products.iloc[:3, :3]

,product_id,product_name,category
0,P001,Wireless Mouse,Electronics
1,P002,Mechanical Keyboard,Electronics
2,P003,USB-C Hub,Electronics


### 2.1 创建副本、设置索引与排序

#### 使用 `copy` 避免修改输入

`copy()` 创建独立的 DataFrame。习题函数通常不应该意外修改调用者传入的数据，因此先创建副本是一种安全习惯：

```python
result = products.copy()
```

后续对 `result` 新增列或赋值，不会改变原始的 `products`。

#### 使用 `set_index` 设置业务键

`set_index('product_id')` 将商品编号从普通列变为行索引，并返回新的 DataFrame。默认情况下，原来的 `product_id` 列会从数据列中移除：

```python
products_by_id = products.set_index('product_id')
products_by_id.loc['P001']
```

如果需要把索引恢复成普通列，可以使用 `reset_index()`。

#### 使用 `sort_values` 排序

单列排序：

```python
df.sort_values(by='unit_price', ascending=False)
```

多列排序时，`by` 和 `ascending` 都传入列表，并且两个列表按位置一一对应：

```python
df.sort_values(
    by=['unit_price', 'product_name'],
    ascending=[False, True],
)
```

这表示先按 `unit_price` **降序**排列；当价格相同时，再按 `product_name` **升序**排列。排序优先级从 `by` 列表的左侧到右侧依次降低。

这些返回新对象的方法可以组成方法链，但长方法链应使用括号和换行保持可读性。

In [6]:
# 复合排序示例：价格降序；价格相同时，商品名称升序
products_sort_example = (
    products.copy()
    .sort_values(
        by=['unit_price', 'product_name'],
        ascending=[False, True],
    )
    .set_index('product_id')
)
products_sort_example[['product_name', 'category', 'unit_price']].head()

,product_name,category,unit_price
product_id,,,
P002,Mechanical Keyboard,Electronics,329
P008,Laptop Stand,Electronics,219
P003,USB-C Hub,Electronics,199
P006,Desk Lamp,Home,159
P001,Wireless Mouse,Electronics,89


In [7]:
products[(products['status'].eq('active')) & (products['unit_price'].le(200))]

,product_id,product_name,category,unit_price,status
0,P001,Wireless Mouse,Electronics,89,active
2,P003,USB-C Hub,Electronics,199,active
3,P004,Notebook,Stationery,18,active
4,P005,Gel Pen Set,Stationery,25,active
6,P007,Water Bottle,Home,79,active


## 习题1：筛选大额折扣订单。

完成 `select_large_discount_orders(orders, min_quantity, min_discount)`。

本题使用从 `data/orders.csv` 读取的 `orders`。讲解示例筛选商品；本题使用相同的筛选、索引和排序方法处理不同的订单字段。

1. 不修改输入 DataFrame；
2. 只保留 `quantity >= min_quantity` 且 `discount >= min_discount` 的订单；
3. 将 `order_id` 设置为索引；
4. 只返回 `order_date`、`customer_id`、`product_id`、`quantity`、`discount` 五列；
5. 先按 `quantity` 降序排列，数量相同时再按 `order_date` 升序排列。

In [ ]:
# UNQ_C1（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: select_large_discount_orders

def select_large_discount_orders(orders, min_quantity, min_discount):
    """筛选达到数量和折扣下限的订单。"""
    # 完成提示：
    # 步骤1：分别对 quantity 和 discount 构造大于等于参数的布尔条件。
    # 步骤2：用 & 组合条件，并用 loc 筛选行和需要的列。
    # 步骤3：把 order_id 设置为索引。
    # 步骤4: 选择需要返回的列，一定要按题目要求的顺序返回列。
    # 步骤5：按 quantity 降序、order_date 升序进行复合排序。
    result = ...

    return result

In [43]:
# 测试习题1
testsuite3.test_select_large_discount_orders(select_large_discount_orders)

恭喜你通过了 test_select_large_discount_orders 测试。1/8


## 3. 缺失值、重复值与数据类型

真实数据经常存在缺失值、重复记录或错误的数据类型。常用检查方法：

- `isna()` / `notna()`：检查缺失值；
- `duplicated(subset=..., keep=...)`：标记重复记录；
- `drop_duplicates(subset=..., keep=...)`：删除重复记录；
- `fillna()`：填补缺失值；
- `astype()`：转换普通数据类型；
- `pd.to_datetime()`：转换日期时间。

`subset='order_id'` 表示只根据订单编号判断重复；`keep='last'` 表示保留最后一次出现的记录。清洗时应先明确业务含义：本实验把折扣缺失解释为“没有折扣”，因此填为 `0.0`；同一订单编号的后记录视为修正版，因此保留最后一条。

数据类型会影响计算和比较：日期字符串应使用 `pd.to_datetime` 转成 `datetime64`，整数数量可以使用 `astype('int64')`。转换后可用 `df.dtypes` 检查结果。

排序不会自动产生连续索引。`reset_index(drop=True)` 会生成从 0 开始的新索引；`drop=True` 防止旧索引被保存成额外一列。完整清洗流程通常是：

```text
copy → 检查 → 填补缺失值 → 去重 → 类型转换 → 排序 → 重置索引
```

这个顺序既保护输入数据，也使函数输出稳定、便于自动测试。

In [10]:
pd.DataFrame({
    '缺失值数量': orders.isna().sum(),
    '数据类型': orders.dtypes.astype(str),
})

,缺失值数量,数据类型
order_id,0,object
order_date,0,object
customer_id,0,object
product_id,0,object
quantity,0,int64
discount,3,float64


In [11]:
orders.loc[orders.duplicated(subset='order_id', keep=False)].sort_values('order_id')

,order_id,order_date,customer_id,product_id,quantity,discount
5,O1006,2026-03-04,C005,P005,4,NaN
6,O1006,2026-03-04,C005,P005,5,0.05


In [12]:
# 清洗演示：使用副本，避免意外修改原始数据
orders_demo = orders.copy()
orders_demo['discount'] = orders_demo['discount'].fillna(0.0)
orders_demo = orders_demo.drop_duplicates(subset='order_id', keep='last')
orders_demo['order_date'] = pd.to_datetime(orders_demo['order_date'])
orders_demo['quantity'] = orders_demo['quantity'].astype('int64')
orders_demo = orders_demo.sort_values('order_id').reset_index(drop=True)
orders_demo.head()

,order_id,order_date,customer_id,product_id,quantity,discount
0,O1001,2026-03-01,C001,P001,2,0.10
1,O1002,2026-03-01,C002,P002,1,0.00
2,O1003,2026-03-02,C003,P004,5,0.05
3,O1004,2026-03-03,C001,P003,1,0.15
4,O1005,2026-03-03,C004,P007,3,0.00


## 习题2：清洗客户基础记录。

完成 `clean_customer_records(customers)`。

本题使用从 `data/customers_dirty.csv` 读取的 `customers`。讲解示例清洗订单；本题复用缺失值、去重、类型转换和排序方法，但使用客户字段和不同参数。

1. 不修改输入 DataFrame；
2. 将 `email` 的缺失值填为 `'unknown'`；
3. 按 `customer_id` 删除重复记录，保留第一条记录；
4. 将 `segment` 转换为 pandas 的 `string` 类型；
5. 按 `customer_id` 降序排列并重置索引；
6. 保持原有列及列顺序不变。

In [13]:
# UNQ_C2（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: clean_customer_records

def clean_customer_records(customers):
    """清理客户记录中的缺失值、重复值和数据类型。"""
    # 完成提示：
    # 步骤1：使用 copy() 创建工作副本。
    # 步骤2：只对 email 列使用 fillna('unknown')。
    # 步骤3：按 customer_id 去重，并注意本题要求 keep='first'。
    # 步骤4：用 astype('string') 转换 segment。
    # 步骤5：按 customer_id 降序排序并 reset_index(drop=True)。
    # 易错提醒：本题不要求清洗字符串空格，也不应填补 city；保持列顺序不变。
    result = ...
    return result

In [14]:
# 测试习题2
testsuite3.test_clean_customer_records(clean_customer_records)

测试失败 test_clean_customer_records: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 4. 向量化字符串清洗

文本字段常包含多余空格、大小写不一致和空字符串。pandas 的 `str` 访问器可以对整列执行向量化字符串操作：

- `str.strip()`：删除首尾空白；
- `str.lower()` / `str.upper()` / `str.title()`：统一大小写；
- `str.contains()`：判断是否包含模式；
- `str.replace()`：替换字符串或正则模式；
- `str.extract()`：用正则表达式提取字段。

对整列使用这些方法通常比逐行循环更简洁，也会自动传播缺失值。

缺失值 `NaN` 和空字符串 `''` 是不同情况，二者都要处理时，可以按下面的顺序组成方法链：

```python
cleaned = (
    series
    .fillna('')       # 先把缺失值变成字符串
    .str.strip()      # 再去除首尾空白
    .replace('', 'Unknown')
    .str.title()
)
```

顺序很重要：只包含空格的字符串要先执行 `str.strip()`，才会变成可以被 `replace('', ...)` 找到的空字符串。对邮箱通常使用 `str.lower()`，对城市名称可以使用 `str.title()`。如果数据中存在重复业务键，应先明确保留第一条还是最后一条，再用 `drop_duplicates` 处理。

In [15]:
customers_raw[['customer_id', 'city', 'segment', 'email']]

,customer_id,city,segment,email
0,C001,shanghai,Student,ALICE@EXAMPLE.COM
1,C002,BEIJING,teacher,bob@example.com
2,C003,SHANGHAI,student,NaN
3,C004,guangzhou,corporate,old-diana@example.com
4,C004,Guangzhou,Corporate,diana@example.com
5,C005,NaN,student,evan@example.com
6,C006,beijing,Teacher,FANG@EXAMPLE.COM
7,C007,guangzhou,corporate,grace@example.com


In [16]:
customers_demo = customers_raw.drop_duplicates('customer_id', keep='last').copy()
customers_demo['customer_name'] = customers_demo['customer_name'].str.strip()
customers_demo['city'] = customers_demo['city'].fillna('').str.strip().replace('', 'Unknown').str.title()
customers_demo['segment'] = customers_demo['segment'].str.strip().str.lower()
customers_demo['email'] = customers_demo['email'].fillna('').str.strip().str.lower().replace('', 'unknown')
customers_demo = customers_demo.sort_values('customer_id').reset_index(drop=True)
customers_demo

,customer_id,customer_name,city,segment,email
0,C001,Alice Zhang,Shanghai,student,alice@example.com
1,C002,Bob Li,Beijing,teacher,bob@example.com
2,C003,Chen Wang,Shanghai,student,unknown
3,C004,Diana Liu,Guangzhou,corporate,diana@example.com
4,C005,Evan Zhou,Unknown,student,evan@example.com
5,C006,Fang Wu,Beijing,teacher,fang@example.com
6,C007,Grace He,Guangzhou,corporate,grace@example.com


## 习题3：统一商品文本格式。

完成 `standardize_product_text(products)`。

本题使用从 `data/products.csv` 读取的 `products`。讲解示例清洗客户姓名、城市和邮箱；本题使用相同的向量化字符串方法处理商品字段。

1. 不修改输入 DataFrame；
2. 按 `product_id` 删除重复记录，保留第一条；
3. `product_name` 删除首尾空白并转换为标题格式；
4. `category` 删除首尾空白并转换为小写；
5. `status` 删除首尾空白并转换为大写；
6. 按 `product_name`、`product_id` 升序排列并重置索引；
7. 保持原有列及列顺序不变。

In [17]:
# UNQ_C3（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: standardize_product_text

def standardize_product_text(products):
    """统一商品名称、类别和状态的文本格式。"""
    # 完成提示：
    # 步骤1：创建副本，并按 product_id 去重，保留第一条。
    # 步骤2：对 product_name 依次使用 str.strip() 和 str.title()。
    # 步骤3：对 category 使用 str.strip() 和 str.lower()。
    # 步骤4：对 status 使用 str.strip() 和 str.upper()。
    # 步骤5：按 product_name、product_id 升序排序并重置索引。
    # 易错提醒：三个文本列的大小写规则不同，不要照搬客户字段的处理规则。
    result = ...
    return result

In [18]:
# 测试习题3
testsuite3.test_standardize_product_text(standardize_product_text)

测试失败 test_standardize_product_text: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 5. 分类数据与分箱

当一列只有少量重复取值时，可以使用 `category` 类型。分类数据内部使用“类别 + 整数编码”表示，通常比重复保存字符串更节省内存，并可加速部分分组运算。

连续数值经常需要分成区间：

- `pd.cut` 按给定边界分箱；
- `pd.qcut` 按样本分位数分箱。

`pd.cut` 的关键参数：

| 参数 | 含义 |
|---|---|
| `bins` | 分箱边界，例如 `[0, 100, 200, np.inf]` |
| `labels` | 每个区间对应的标签，数量比边界少 1 |
| `right=True` | 区间包含右端点，这是默认设置 |
| `include_lowest=True` | 第一个区间包含最小边界 |
| `ordered=True` | 生成有顺序的分类类型 |

当 `bins=[0, 100, 200, np.inf]`、`right=True` 时，区间可理解为 `[0, 100]`、`(100, 200]`、`(200, +∞]`，所以价格 `100` 属于第一档，`200` 属于第二档。

`DataFrame.assign(新列=...)` 会返回包含新列的 DataFrame，不修改原对象，并把新列放在现有列之后。可以用 `series.dtype` 检查是否为分类类型，用 `series.cat.categories` 和 `series.cat.ordered` 查看类别及其顺序。

In [19]:
price_bins = [0, 100, 200, np.inf]
price_labels = ['低价', '中价', '高价']

products_with_band = products.assign(
    price_band=pd.cut(
        products['unit_price'],
        bins=price_bins,
        labels=price_labels,
        include_lowest=True,
        ordered=True,
    )
)
products_with_band[['product_id', 'unit_price', 'price_band']]

,product_id,unit_price,price_band
0,P001,89,低价
1,P002,329,高价
2,P003,199,中价
3,P004,18,低价
4,P005,25,低价
5,P006,159,中价
6,P007,79,低价
7,P008,219,高价


In [20]:
products_with_band['price_band'].dtype, products_with_band['price_band'].cat.categories

(CategoricalDtype(categories=['低价', '中价', '高价'], ordered=True, categories_dtype=object),
 Index(['低价', '中价', '高价'], dtype='object'))

## 习题4：划分订单规模。

完成 `add_order_size_band(orders)`。

本题使用从 `data/orders.csv` 读取的 `orders`。讲解示例根据商品价格分箱；本题仍使用 `pd.cut`，但改为根据订单数量和新的边界划分规模。

1. 不修改输入 DataFrame；
2. 使用 `pd.cut` 新增有序分类列 `order_size`；
3. 分箱边界为 `[0, 1, 3, 正无穷]`；
4. 标签依次为 `'单件'`、`'小批'`、`'大批'`；
5. 包含最低值，且 `1` 属于单件、`3` 属于小批；
6. 返回原有全部列，并把 `order_size` 放在最后。

In [21]:
# UNQ_C4（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: add_order_size_band

def add_order_size_band(orders):
    """根据商品数量为订单添加有序规模档位。"""
    # 完成提示：
    # 步骤1：创建 orders 的副本。
    # 步骤2：为 quantity 准备边界 [0, 1, 3, np.inf] 和三个新标签。
    # 步骤3：调用 pd.cut，并设置 include_lowest=True、right=True、ordered=True。
    # 步骤4：把结果写入副本的新列 order_size。
    # 易错提醒：这里的分箱列、边界和标签都不同于商品价格示例；
    #          自动测试会检查 1、3 以及边界两侧的值。
    result = ...
    return result

In [22]:
# 测试习题4
testsuite3.test_add_order_size_band(add_order_size_band)

测试失败 test_add_order_size_band: 'ellipsis' object has no attribute 'columns'


## 6. 使用 merge 连接数据表

关系型数据通常拆分为事实表和维度表。例如订单表只保存 `product_id` 和 `customer_id`，商品名称、价格和客户城市分别存放在其他表中。

`pd.merge` 根据键连接数据，类似 SQL JOIN：

- `how='inner'`：只保留两边都匹配的键；
- `how='left'`：保留左表全部记录，右表未匹配字段为缺失值；
- `how='right'`：保留右表全部记录；
- `how='outer'`：保留两边键的并集。

应显式指定 `on`，并使用 `validate` 检查连接基数：

| `validate` | 含义 |
|---|---|
| `'one_to_one'` | 两边连接键都必须唯一 |
| `'one_to_many'` | 左键唯一，右键可以重复 |
| `'many_to_one'` | 左键可以重复，右键必须唯一 |
| `'many_to_many'` | 两边都可以重复，不做唯一性约束 |

订单中同一商品可以出现多次，而商品表中每个 `product_id` 应只有一行，因此订单连接商品是 `many_to_one`。如果商品表意外出现重复编号，关系校验会立即报错，避免连接后行数无声膨胀。

连接前只选择需要的维度列，例如：

```python
products[['product_id', 'product_name', 'category', 'unit_price']]
```

这样可以减少无关列，并避免同名列产生 `_x`、`_y` 后缀。连续调用两次 `merge` 可以依次补充商品和客户信息。

连接后可以直接使用整列运算生成派生列：

```python
detail['revenue'] = (
    detail['quantity'] * detail['unit_price'] * (1 - detail['discount'])
).round(2)
```

`round(2)` 将结果保留两位小数。最后使用 `detail[[...]]` 按列表顺序选择列，就能同时控制保留哪些列以及返回的列顺序；再用 `sort_values(...).reset_index(drop=True)` 获得稳定的行顺序。

In [23]:
# 准备用于连接的干净副本
orders_ready = orders.drop_duplicates('order_id', keep='last').copy()
orders_ready['discount'] = orders_ready['discount'].fillna(0.0)
orders_ready['order_date'] = pd.to_datetime(orders_ready['order_date'])
orders_ready['quantity'] = orders_ready['quantity'].astype('int64')

customers_ready = customers_raw.drop_duplicates('customer_id', keep='last').copy()
customers_ready['customer_name'] = customers_ready['customer_name'].str.strip()
customers_ready['city'] = customers_ready['city'].fillna('').str.strip().replace('', 'Unknown').str.title()

sales_example = orders_ready.merge(
    products[['product_id', 'product_name', 'category', 'unit_price']],
    on='product_id', how='left', validate='many_to_one',
).merge(
    customers_ready[['customer_id', 'customer_name', 'city']],
    on='customer_id', how='left', validate='many_to_one',
)
sales_example['revenue'] = (
    sales_example['quantity'] * sales_example['unit_price'] * (1 - sales_example['discount'])
).round(2)
sales_example.head()

,order_id,order_date,customer_id,product_id,quantity,discount,product_name,category,unit_price,customer_name,city,revenue
0,O1001,2026-03-01,C001,P001,2,0.10,Wireless Mouse,Electronics,89,Alice Zhang,Shanghai,160.20
1,O1002,2026-03-01,C002,P002,1,0.00,Mechanical Keyboard,Electronics,329,Bob Li,Beijing,329.00
2,O1003,2026-03-02,C003,P004,5,0.05,Notebook,Stationery,18,Chen Wang,Shanghai,85.50
3,O1004,2026-03-03,C001,P003,1,0.15,USB-C Hub,Electronics,199,Alice Zhang,Shanghai,169.15
4,O1005,2026-03-03,C004,P007,3,0.00,Water Bottle,Home,79,Diana Liu,Guangzhou,237.00


In [24]:
# 左连接保留了未知客户 C999 的订单，未匹配的客户字段为缺失值
sales_example.loc[sales_example['customer_name'].isna()]

,order_id,order_date,customer_id,product_id,quantity,discount,product_name,category,unit_price,customer_name,city,revenue
11,O1012,2026-03-07,C999,P003,1,0.0,USB-C Hub,Electronics,199,NaN,NaN,199.0


## 习题5：构建库存销售明细。

完成 `build_stock_sales_detail(inventory, monthly_sales)`。

本题使用 `data/inventory_shanghai.csv` 和 `data/monthly_sales_wide.csv`。讲解示例连接订单、商品和客户表；本题改为连接库存与月度销售，并使用不同的连接方式和基数校验。

1. 不修改任何输入 DataFrame；
2. 从 `monthly_sales` 只选择 `product_id` 和 `2026-03`；
3. 以 `product_id` 为键执行内连接，并设置一对一关系校验；
4. 新增 `sales_per_stock = 2026-03销售额 ÷ stock`，保留两位小数；
5. 返回列顺序为 `product_id`、`warehouse`、`stock`、`2026-03`、`sales_per_stock`；
6. 按 `sales_per_stock` 降序、`product_id` 升序排列并重置索引。

In [25]:
# UNQ_C5（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: build_stock_sales_detail

def build_stock_sales_detail(inventory, monthly_sales):
    """连接库存和三月销售数据并计算单位库存销售额。"""
    # 完成提示：
    # 步骤1：只选择 monthly_sales 的 product_id 和 2026-03 两列。
    # 步骤2：使用 merge，以 product_id 为键执行 inner 连接，
    #        并设置 validate='one_to_one'。
    # 步骤3：使用整列运算计算 sales_per_stock，并 round(2)。
    # 步骤4：显式选择五个返回列，再按题目要求复合排序并重置索引。
    # 易错提醒：本题不是 left 连接，也不是 many_to_one；不要连接讲解示例中的客户字段。
    result = ...
    return result

In [26]:
# 测试习题5
testsuite3.test_build_stock_sales_detail(build_stock_sales_detail)

测试失败 test_build_stock_sales_detail: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 7. groupby：拆分、应用、合并

`groupby` 的思路可以概括为：

1. 按键把数据拆分成组；
2. 对每组执行聚合或转换；
3. 把结果合并成新的 pandas 对象。

常用聚合包括 `sum`、`mean`、`count`、`nunique`、`min` 和 `max`。其中：

- `count` 统计非缺失记录数；
- `size` 统计组内总行数，包括缺失值；
- `nunique` 统计不同值的数量，适合计算唯一订单数。

`groupby('category', as_index=False)` 会让分组键 `category` 保持为普通列，省去之后的 `reset_index`。命名聚合的格式是：

```python
输出列名=('输入列名', '聚合函数')
```

例如：

```python
.agg(
    order_count=('order_id', 'nunique'),
    total_quantity=('quantity', 'sum'),
)
```

聚合结果仍可继续进行列计算、`round`、复合排序和索引重置。若按多列分组且不设置 `as_index=False`，结果会产生层次化索引，可以使用 `reset_index()` 将各级索引恢复为普通列。

In [27]:
category_summary_example = (
    sales_example
    .groupby('category', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_quantity=('quantity', 'sum'),
        total_revenue=('revenue', 'sum'),
    )
)
category_summary_example['total_revenue'] = category_summary_example['total_revenue'].round(2)
category_summary_example = (
    category_summary_example
    .sort_values(
        by=['total_revenue', 'category'],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)
category_summary_example

,category,order_count,total_quantity,total_revenue
0,Electronics,7,8,1406.65
1,Home,2,5,395.00
2,Stationery,3,20,375.25


In [28]:
# 多列分组会形成层次化索引；reset_index 可将索引层级恢复为普通列
sales_example.groupby(['category', 'city'])['revenue'].sum().reset_index().head()

,category,city,revenue
0,Electronics,Beijing,615.10
1,Electronics,Shanghai,592.55
2,Home,Guangzhou,237.00
3,Home,Shanghai,158.00
4,Stationery,Guangzhou,171.00


## 习题6：按客户汇总订单。

完成 `summarize_customer_orders(orders)`。

本题使用从 `data/orders.csv` 读取的 `orders`。讲解示例按商品类别汇总收入；本题使用相同的 `groupby` 和命名聚合语法，改为按客户汇总订单字段。

1. 不修改输入 DataFrame；
2. 按 `customer_id` 分组；
3. 计算唯一订单数 `order_count`；
4. 计算不同商品数 `product_count`；
5. 计算商品数量合计 `total_quantity`；
6. 计算平均折扣 `average_discount`，并保留三位小数；
7. 返回列顺序为 `customer_id`、`order_count`、`product_count`、`total_quantity`、`average_discount`；
8. 按 `total_quantity` 降序、`customer_id` 升序排列并重置索引。

In [29]:
# UNQ_C6（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: summarize_customer_orders

def summarize_customer_orders(orders):
    """按客户汇总唯一订单、商品、数量和平均折扣。"""
    # 完成提示：
    # 步骤1：按 customer_id 分组，并让分组键保持为普通列。
    # 步骤2：使用命名聚合：order_id 和 product_id 使用 nunique，
    #        quantity 使用 sum，discount 使用 mean。
    # 步骤3：对 average_discount 使用 round(3)。
    # 步骤4：按 total_quantity 降序、customer_id 升序排序并重置索引。
    # 易错提醒：这里汇总的是客户订单，不需要 category、revenue 或销售收入字段。
    result = ...
    return result

In [30]:
# 测试习题6
testsuite3.test_summarize_customer_orders(summarize_customer_orders)

测试失败 test_summarize_customer_orders: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 8. concat：沿轴拼接数据

`merge` 按键横向连接字段；`concat` 则沿某个轴直接拼接对象：

- `axis='index'` 或 `axis=0`：纵向追加行，最常见；
- `axis='columns'` 或 `axis=1`：横向追加列；
- `ignore_index=True`：忽略各表原索引并生成连续索引；
- `keys`：给不同数据来源添加外层索引；
- `join='outer'`：保留所有列，这是默认行为；
- `join='inner'`：只保留各表共有的列。

纵向拼接会按**列名**对齐，而不是按列的位置对齐。如果某张表缺少某列，拼接结果的相应位置会产生缺失值。因此拼接前要确认各表的列含义和数据类型一致。

`pd.concat` 返回新对象，不会修改输入表。本实验中的两个仓库使用相同的 `product_id`、`warehouse`、`stock` 字段，适合先纵向拼接，再用上一节的 `groupby`、`sum` 和 `nunique` 汇总每个商品的总库存与仓库数。汇总后的复合排序方法与习题 1 相同。

In [31]:
all_inventory = pd.concat(
    [inventory_shanghai, inventory_beijing],
    axis='index',
    ignore_index=True,
)
all_inventory

,product_id,warehouse,stock
0,P001,Shanghai,30
1,P002,Shanghai,12
2,P003,Shanghai,8
3,P004,Shanghai,100
4,P005,Shanghai,80
5,P003,Beijing,15
6,P004,Beijing,60
7,P006,Beijing,10
8,P007,Beijing,20
9,P008,Beijing,14


In [32]:
inventory_summary_example = (
    all_inventory
    .groupby('product_id', as_index=False)
    .agg(
        total_stock=('stock', 'sum'),
        warehouse_count=('warehouse', 'nunique'),
    )
    .sort_values(['total_stock', 'product_id'], ascending=[False, True])
    .reset_index(drop=True)
)
inventory_summary_example

,product_id,total_stock,warehouse_count
0,P004,160,2
1,P005,80,1
2,P001,30,1
3,P003,23,2
4,P007,20,1
5,P008,14,1
6,P002,12,1
7,P006,10,1


## 习题7：合并订单批次并汇总。

完成 `summarize_order_batches(first_batch, second_batch)`。

两个参数采用 `data/orders.csv` 的相同字段结构，表示分别到达的两批订单。讲解示例纵向拼接两个仓库的库存；本题复用 `concat` 和分组汇总，但处理订单字段。

1. 不修改两个输入 DataFrame；
2. 使用 `pd.concat` 纵向拼接两批订单并重置索引；
3. 按 `customer_id` 分组；
4. 计算唯一订单数 `order_count`；
5. 计算商品数量合计 `total_quantity`；
6. 返回列顺序为 `customer_id`、`order_count`、`total_quantity`；
7. 按 `total_quantity` 降序、`customer_id` 升序排列并重置索引。

In [33]:
# UNQ_C7（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: summarize_order_batches

def summarize_order_batches(first_batch, second_batch):
    """拼接两批订单并按客户汇总。"""
    # 完成提示：
    # 步骤1：使用 pd.concat 纵向拼接两个参数，并设置 ignore_index=True。
    # 步骤2：按 customer_id 分组。
    # 步骤3：使用命名聚合计算 order_id 的 nunique 和 quantity 的 sum。
    # 步骤4：选择并排列三个返回列，再复合排序并重置索引。
    # 易错提醒：本题的数据列来自订单，不再使用 product_id、warehouse 或 stock 汇总库存。
    result = ...
    return result

In [34]:
# 测试习题7
testsuite3.test_summarize_order_batches(summarize_order_batches)

测试失败 test_summarize_order_batches: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 9. 数据重塑：宽表与长表

同一组数据可以有不同形状：

- **宽表**：每个月份是一列，便于阅读和矩阵计算；
- **长表**：每一行是一条“商品—月份—销售额”观察，便于分组、绘图和数据库存储。

`melt` 将宽表转成长表，重要参数包括：

| 参数 | 含义 |
|---|---|
| `id_vars` | 转换时保持不动的标识列，例如 `product_id` |
| `value_vars` | 要合并的值列；省略时使用所有非标识列 |
| `var_name` | 原列名合并后所在的新列名称，例如 `month` |
| `value_name` | 原单元格数值所在的新列名称，例如 `sales` |

调用：

```python
wide.melt(
    id_vars='product_id',
    var_name='month',
    value_name='sales',
)
```

会返回新的 DataFrame，不修改 `wide`。`melt` 的结果顺序取决于原表列顺序；若函数输出要求稳定顺序，应继续使用 `sort_values` 和 `reset_index(drop=True)`。

反向操作主要有：

- `pivot`：长表转宽表，要求“行键 + 列键”的组合唯一；
- `pivot_table`：允许重复组合，并通过聚合函数处理重复值；
- `stack`：把列索引层级移动到行索引；
- `unstack`：把行索引层级移动到列索引。

如果只希望返回固定三列，可用 `result[['product_id', 'month', 'sales']]` 明确列顺序。

In [35]:
monthly_sales_wide

,product_id,2026-01,2026-02,2026-03
0,P001,1200,1350,1280
1,P002,980,1120,1460
2,P003,1560,1490,1720
3,P004,620,710,680
4,P005,540,590,640
5,P007,810,760,920
6,P008,1340,1410,1530


In [36]:
monthly_sales_long = monthly_sales_wide.melt(
    id_vars='product_id',
    var_name='month',
    value_name='sales',
).sort_values(['product_id', 'month']).reset_index(drop=True)
monthly_sales_long.head(8)

,product_id,month,sales
0,P001,2026-01,1200
1,P001,2026-02,1350
2,P001,2026-03,1280
3,P002,2026-01,980
4,P002,2026-02,1120
5,P002,2026-03,1460
6,P003,2026-01,1560
7,P003,2026-02,1490


In [37]:
# pivot 可以把长表恢复为宽表
monthly_sales_long.pivot(
    index='product_id',
    columns='month',
    values='sales',
).reset_index()

month,product_id,2026-01,2026-02,2026-03
0,P001,1200,1350,1280
1,P002,980,1120,1460
2,P003,1560,1490,1720
3,P004,620,710,680
4,P005,540,590,640
5,P007,810,760,920
6,P008,1340,1410,1530


## 习题8：商品属性宽表转长表。

完成 `reshape_product_attributes(products)`。

本题使用从 `data/products.csv` 读取的 `products`。讲解示例将月份列转换为 `month` 和 `sales`；本题使用相同的 `melt` 方法转换商品文本属性，并改变标识列和输出列名。

1. 不修改输入 DataFrame；
2. 使用 `melt`，保留 `product_id` 和 `unit_price` 作为标识列；
3. 只转换 `product_name`、`category`、`status` 三列；
4. 原列名保存到 `attribute`，对应的值保存到 `value`；
5. 返回列顺序为 `product_id`、`unit_price`、`attribute`、`value`；
6. 按 `product_id`、`attribute` 升序排列并重置索引。

In [38]:
# UNQ_C8（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: reshape_product_attributes

def reshape_product_attributes(products):
    """把商品的多个属性列转换成长表。"""
    # 完成提示：
    # 步骤1：调用 melt，把 product_id、unit_price 作为 id_vars。
    # 步骤2：把 product_name、category、status 作为 value_vars。
    # 步骤3：设置 var_name='attribute'、value_name='value'。
    # 步骤4：显式选择四个返回列，按 product_id、attribute 排序并重置索引。
    # 易错提醒：本题转换的是商品属性，不要使用讲解示例中的 month 或 sales 列名。
    result = ...
    return result

In [39]:
# 测试习题8
testsuite3.test_reshape_product_attributes(reshape_product_attributes)

测试失败 test_reshape_product_attributes: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 10. 实验讨论

请结合代码和运行结果讨论：

1. 为什么 pandas 按索引自动对齐既方便又可能产生意外的缺失值？
   
2. 删除缺失值、填补缺失值分别适合什么情况？不恰当处理会造成什么偏差？
   
3. `merge` 和 `concat` 的用途有什么本质区别？为什么表连接时要检查键的唯一性？
   
4. 长表和宽表分别适合哪些任务？请各举一个例子。

## 11. 生成并提交实验报告

运行下面两个单元格前，请确认：

- 已填写顶部学生信息和有效 Email；
- 已保存 Notebook；
- 八个习题测试均已运行；
- 在实验室环境中可以访问 LabDrop 服务。

非实验室环境需要补交时，请将 PDF 提交到对应班级和实验的收集表。

In [40]:
import importlib
import notebook2pdf
importlib.reload(notebook2pdf)

stu_grade = testsuite3.grade_all_tests(notebook_file)
pdf_file = f"{stu_info['class_id']}-{stu_info['student_id']}-{stu_info['name']}-实验报告3-{stu_grade}.pdf"
notebook2pdf.convert_notebook_to_webpdf(notebook_file, pdf_file)

正在从 notebook 文件收集测试函数: /Users/zhoujason/Desktop/Projects/data-modeling-course-student/实验03-pandas/实验3-pandas.ipynb
测试失败 test_select_large_discount_orders: DataFrame.columns are different

DataFrame.columns values are different (100.0 %)
[left]:  Index(['customer_id', 'product_id', 'quantity', 'discount', 'order_date'], dtype='object')
[right]: Index(['order_date', 'customer_id', 'product_id', 'quantity', 'discount'], dtype='object')
At positional index 0, first diff: customer_id != order_date
测试失败 test_clean_customer_records: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead
测试失败 test_standardize_product_text: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead
测试失败 test_add_order_size_band: 'ellipsis' object has no attribute 'columns'
测试失败 test_build_stock_sales_detail: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead
测试失败 test_summarize_customer_ord

Notebook to PDF: 100%|██████████| 100/100 [00:04<00:00, 20.26%/s]



成功生成报告文件: 25智能01-未填写-未填写-实验报告3-0.pdf


In [41]:
from report_submission import submit_pdf_report

receipt = submit_pdf_report(
    pdf_file,
    stu_info,
    experiment_number=3,
)

receipt

ReportSubmissionError: Server returned HTTP 502

## 参考资料

- Wes McKinney, *Python for Data Analysis, 3rd Edition*：第 5 章 Getting Started with pandas、第 7 章 Data Cleaning and Preparation、第 8 章 Data Wrangling: Join, Combine, and Reshape。
- pandas 官方文档：https://pandas.pydata.org/docs/